# 01 — Canonical Verdicts: Cross-family ingestion

**Abstract.** This notebook is the **reproducibility companion** to the
stochastic-FX variance-proxy working paper at
`docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex`. Three
trios will progressively render the paper's §7 results:

- **Trio 1 (this dispatch)** — load and cross-check the three
  authoritative canonical-pin `InversionVerdict` JSON sidecars (GBM, OU,
  Merton) emitted by the Task-5 IO Boundary classes; reproduce paper
  §7.1's cross-family summary table.
- Trio 2 — per-family interpretation figures (paper §7.2).
- Trio 3 — strip-preservation + determinism + Wave-2 summary (paper
  §7.3–§7.5).

The notebook is **verify-only** at runtime: it reads the committed
JSON sidecars under `simulations/stochastic_fx/data/` via the
`InversionVerdictEmitter().read(family_id)` round-trip API and does
**not** re-run the Phase B Monte-Carlo or Phase C KS dispatch. Re-running
those would touch the source-of-truth verdicts and is out of scope.

**Reproducibility pin.** All numerics here are anchored to
`CANONICAL_RNG_SEED = 42` (the paper §7 anchor) — see
`simulations/notebooks/stochastic_fx/env.py`.

**Decision-citation block — canonical-pin verdict source**

- **Reference**: `docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex` §1 (introduction) + §7.1 (cross-family results table); `docs/specs/2026-05-11-stochastic-fx-variant-design.md` v0.5 §4.2 (canonical pins) and §5 (Pin Z1.x anti-fishing invariants).
- **Why used**: The committed per-family `inversion_verdict_{family_id}.json` sidecars under `simulations/stochastic_fx/data/` are the authoritative single-source-of-truth for the canonical-pin verification verdicts. Spec v0.5 §5 Pin Z1.5 explicitly bans recomputation-from-source in any downstream consumer — the JSON sidecars are the audit-anchored artefact, and only an `InversionVerdictEmitter().read(...)` round-trip is permitted (which re-applies the frozen-dc invariants in `InversionVerdict.__post_init__`).
- **Relevance to results**: If this decision were different — e.g., if the notebook re-ran the Phase B Monte-Carlo locally — the per-family rel-errs and KS p-values would drift across executions (Phase C has finite-sample noise), the paper §7.1 cross-family table would no longer be byte-stable, and Pin Z1.5 anti-fishing would be violated. Reading the sidecars via the emitter API instead preserves the audit-block hex prefixes (paper §7.1 column 7) and gives Type-I-rejection nullity at fixed `seed = 42`.
- **Connection to simulator**: Maps to `simulations/stochastic_fx/emit.py::InversionVerdictEmitter` (IO Boundary tier) — the only sanctioned reader of the JSON sidecars; its `read(family_id)` method validates `schema_version == 'v1.0'` and reconstructs the frozen-dc `InversionVerdict` via `simulations/stochastic_fx/types.py::InversionVerdict`.

In [ ]:
from __future__ import annotations

import pandas as pd

from simulations.notebooks.stochastic_fx.env import (
    CANONICAL_RNG_SEED,
    DATA_ROOT,
    reproducibility_seed,
)
from simulations.stochastic_fx import (
    InversionVerdictEmitter,
    KS_PVALUE_FLOOR,
    MOMENT_REL_TOL,
    NUMERICAL_IDENTITY_TOL,
)

_seed = reproducibility_seed()
print(f"canonical rng seed: {_seed} (env.CANONICAL_RNG_SEED = {CANONICAL_RNG_SEED})")
print(f"data root         : {DATA_ROOT}")
print(
    f"tolerances        : phase_a = {NUMERICAL_IDENTITY_TOL}, "
    f"phase_b = {MOMENT_REL_TOL}, phase_c = {KS_PVALUE_FLOOR}"
)

## Trio 1 — Cross-family verdict ingestion

**Why this trio.** Paper §7.1 reports a single cross-family summary table
consolidating the three canonical-pin verification verdicts (GBM, OU,
Merton). The table is the entry point to §7.2's per-family interpretation
and is the visible artefact a reader of the paper checks first. This
trio loads the three `InversionVerdict` JSON sidecars — the authoritative
canonical-pin verdict source per spec v0.5 §5 (Pin Z1.x anti-fishing) —
and reconstructs that table from a pandas DataFrame indexed by family.

The sidecars are read via
`InversionVerdictEmitter().read(family_id)` rather than `json.load`
directly: the emitter API re-applies the schema-v1.0 invariants (missing
keys → `RoundTripDriftError`; bad schema_version → `RoundTripDriftError`)
and runs the frozen-dc `InversionVerdict.__post_init__` invariants
(composite-AND check on the three phase-pass flags, finite-float checks
on the residuals / rel-errs / p-value). Any drift in the on-disk JSON
fails fast here rather than silently producing a stale table.

**Reference**: paper `docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex` §7.1 (table layout); `notes/STOCHASTIC_FX_RESULTS.md` §1 (same table rendered as markdown); spec v0.5 §6 (three-phase verification semantics).

In [ ]:
# Load the three canonical-pin verdicts via the sanctioned IO Boundary
# reader. The emitter is configured with the project DATA_ROOT (not the
# emitter's default relative path) so this notebook is location-agnostic.
emitter = InversionVerdictEmitter(emit_dir=DATA_ROOT)

FAMILY_IDS: tuple[str, ...] = ("gbm", "ou", "merton")
FAMILY_LABELS: dict[str, str] = {
    "gbm": "GBM",
    "ou": "OU",
    "merton": "Merton",
}

verdicts = {fid: emitter.read(fid) for fid in FAMILY_IDS}

# Build the paper §7.1 cross-family summary DataFrame.
rows: list[dict[str, object]] = []
for fid in FAMILY_IDS:
    v = verdicts[fid]
    rows.append(
        {
            "family": FAMILY_LABELS[fid],
            "composite_pass": v.composite_pass,
            "phase_a_max_residual": v.phase_a_max_residual,
            "phase_b_mean_rel_err": v.phase_b_mean_rel_err,
            "phase_b_var_rel_err": v.phase_b_var_rel_err,
            "phase_c_ks_pvalue": v.phase_c_ks_pvalue,
            "audit_block_prefix": v.audit_block[:16],
        }
    )

df = pd.DataFrame(rows).set_index("family")

# Headless-stable rendering — print the full string repr.
print(df.to_string())

**Interpretation — per-family numerics at `seed = 42`.**

Reading column by column against the paper §7.1 thresholds:

- **`composite_pass`** — all three families PASS, matching paper §7.1
  row-1 verdicts. The composite-AND invariant (`phase_a_pass AND
  phase_b_pass AND phase_c_pass`) is enforced in
  `InversionVerdict.__post_init__`, so a `True` here is
  emitter-validated.
- **`phase_a_max_residual`** — algebraic-identity residuals are at
  machine epsilon for all three families (GBM ≈ 7e-15, OU ≈ 4e-15,
  Merton ≈ 2e-13), all far below `NUMERICAL_IDENTITY_TOL = 1e-06`.
  Merton is two orders larger than GBM/OU because the compound-Poisson
  jump term contributes additional float-arithmetic accumulations;
  still 7 orders below the gate.
- **`phase_b_mean_rel_err`** — Phase B mean-only gate per spec v0.5
  §11.6 (Pin Z1.3b). Observed: GBM ≈ 1.12 %, OU ≈ 0.17 %, Merton ≈
  0.70 %. All three are comfortably below `MOMENT_REL_TOL = 0.05`
  (5 %); the OU value is smallest because the Vasicek-style exact
  transition density carries the lowest MC-sampling noise on the σ_T
  mean. This is the gate that drives `phase_b_pass`.
- **`phase_b_var_rel_err`** — paper §11.6 audit-trail observation
  only: NOT a gate. Observed: GBM ≈ 0.213, OU ≈ 0.027, Merton ≈ 2.73.
  Merton's value is roughly 100× OU's because Merton's high jump
  kurtosis inflates the finite-sample variance estimator at `N_PATHS =
  1000`; the disposition under Pin Z1.3b is to surface this number
  rather than gate on it (gating would force a non-canonical
  variance-shrinkage on the jump-diffusion family — anti-fishing per
  spec v0.5 §11.6).
- **`phase_c_ks_pvalue`** — Phase C KS goodness-of-fit floor per spec
  v0.5 §11.7 (Pin Z1.4). Observed: GBM ≈ 0.123, OU ≈ 0.303, Merton ≈
  0.116. All three are comfortably above `KS_PVALUE_FLOOR = 0.01`, so
  there is **no Type-I rejection at `seed = 42`** on any family.
  Merton uses the empirical-CDF reference dispatch per Pin Z1.4 (high-N
  reference run at `N_REF = 100_000`, `N_REF_SEED = 20260513`); GBM and
  OU use their closed-form CDFs.
- **`audit_block_prefix`** — the first 16 hex chars of the
  SHA-256 audit block bound to `(canonical_params, rng_seed,
  schema_version)`. These prefixes are paper §7.1 column 7 and are
  the per-family lineage anchors.

**Reference**: paper `docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex` §7.1 (cross-family table values), §11.6 (Phase B variance rel-err audit-trail disposition), §11.7 (Phase C per-family reference dispatch); spec v0.5 §11.6–§11.7 (Pin Z1.3b and Pin Z1.4).

---

**HALT — end of Trio 1.** Per `memory/feedback_notebook_trio_checkpoint.md`,
the next trio (per-family interpretation figures) is a separate dispatch
after human review of this DataFrame.